# 학습 영상에서 가까운 골반을 짚을 수 있는가?
앞선 체간 실험에서는 a05와 b02의 골반 좌표를 바꾸면 이륙부터 무릎 통과까지의 각도 변화 방향도 바뀌었다. 이번에는 실제 원판 베이스라인 학습 영상에서 가까운 쪽 고관절 소켓을 다시 짚을 만한 장면이 몇 개인지 살핀다.

**추정 후보**는 엉덩이의 바깥 윤곽과 허벅지 방향을 함께 보고 위치를 좁힐 수 있는 장면이다. **판독 어려움**은 옷·기구·흐림 때문에 한 점을 고르기 어려운 장면이다. 소켓은 몸 안에 있다. 옷 경계나 주름을 관절 자체로 취급하면 같은 지점을 반복해서 짚어도 해부학적 위치는 틀릴 수 있다.

## 1. 어떤 프레임을 세었나
목록은 실제 학습 실행의 사람·영상 기록, 기존 사람별 분할, 학습 이미지의 프레임 번호와 대조한다. CSV의 좌표는 이번 예비 판독에만 쓴 원본 픽셀이다. 프레임 번호는 0부터 센다.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from wlcv.config import DATASET, ROOT, RUNS, SPLIT, VIDEOS
from wlcv.evaluation.hip_visibility import plot_review

audit = pd.read_csv(ROOT / "data/metadata/train_hip_visibility.csv")
manifest = pd.read_csv(RUNS / "plate_baseline/train_people.csv")
split = pd.read_csv(SPLIT)
metadata = pd.read_csv(ROOT / "data/metadata/videos.csv").set_index("video_id")
output = ROOT / "outputs/pose_baseline/train_hip_visibility"
output.mkdir(parents=True, exist_ok=True)

keys = ["video_id", "person_id"]
trained = set(manifest[keys].itertuples(index=False, name=None))
assert trained == set(audit[keys].itertuples(index=False, name=None))
assert trained <= set(split.loc[split.role == "train", keys].itertuples(index=False, name=None))
assert not audit.duplicated(["video_id", "frame_index"]).any()
assert audit.near_side.eq(audit.video_id.map(metadata.near_side)).all()

plate_frames = audit.loc[audit.in_plate_train == 1].copy()
saved_ids = set()
for path in (DATASET / "images/train").glob("*.jpg"):
    video, number = path.stem.rsplit("_f", 1)
    saved_ids.add((video, int(number)))
assert saved_ids == set(plate_frames[["video_id", "frame_index"]].itertuples(index=False, name=None))

events = audit.loc[audit.phase.notna()].copy()
display(events[["person_id", "near_side", "frame_index", "phase", "phase_source"]])

## 2. 골반을 짚을 단서가 얼마나 있었나
첫 표는 원판이 달린 6편의 이륙 후보와 무릎 통과를 센다. 이륙은 원판이 바닥에서 움직이기 시작하는 앞뒤를 다시 본 후보이고, 무릎 통과는 5편에서 기존 1차 라벨을 썼다. c05의 무릎 통과 16번은 원본에서 고른 잠정 시점이다.

둘째 표는 베이스라인에 들어간 학습 이미지 전체를 한 번씩 확인한 결과다. 원판 없는 b05·b06도 포함한다. 두 표에는 같은 프레임이 들어갈 수 있다.

In [ ]:
categories = ["추정 후보", "판독 어려움"]
by_phase = pd.crosstab(events.phase, events.screen).reindex(
    index=["이륙 후보", "무릎 통과"], columns=categories, fill_value=0
)
by_person = pd.crosstab(plate_frames.person_id, plate_frames.screen).reindex(columns=categories)
display(by_phase, by_person)
print("두 표에 공통으로 들어간 프레임:", events.in_plate_train.sum())
print("이벤트 장면의 가까운 쪽별 분포")
display(pd.crosstab(events.near_side, events.screen).reindex(columns=categories))

## 3. 같은 프레임의 두 추정점이 얼마나 떨어졌나
같은 판독 주체가 두 차례에 영상 ID와 이전 점을 숨기고, 장면 순서와 격자 원점을 바꿔 좌표를 추정했다. 트레이너의 반복 라벨은 아직 없다. 1·2차 모두 위치를 고르기 어려웠던 장면은 거리를 비워 둔다. 아래 거리는 같은 원본 프레임 안의 두 점 사이 거리다. 영상 해상도와 촬영 배율이 달라 서로 다른 선수에게 하나의 픽셀 허용치를 적용하지 않는다.

In [ ]:
events["repeat_distance_px"] = np.hypot(
    events.read2_x - events.read1_x, events.read2_y - events.read1_y
).round(1)
columns = ["person_id", "near_side", "phase", "frame_index", "screen", "repeat_distance_px", "visual_cue"]
display(events[columns])
events.to_csv(output / "phase_audit.csv", index=False)
by_person.to_csv(output / "plate_train_counts.csv")

## 4. 원본에서 어디를 짚었나
위는 이륙 후보, 아래는 무릎 통과다. 청록색 십자는 첫 판독, 주황색 원은 둘째 판독이다. 두 점이 가까워도 실제 소켓 위치가 확인된 것은 아니다. 표시를 숨기려면 `show_points=False`로 바꾸고, 한 사람을 크게 보려면 `people=["a01"]`처럼 선택한다.

In [ ]:
people = ["a01", "a06", "b01", "b04", "c02", "c05"]
show_points = True
fig = plot_review(events, people, metadata.filename, VIDEOS, show_points=show_points)
if show_points and people == sorted(events.person_id.unique()):
    fig.savefig(output / "hip_visibility_review.png", dpi=125, bbox_inches="tight")
plt.show()

## 다음 판독
b04의 두 장과 두 추정점 차이가 컸던 a01 이륙부터, 트레이너가 기존 표시를 숨기고 다시 짚어 본다. a06과 c05는 보류 사례로 남긴다. 가까운 쪽이 오른쪽인 장면에서 라벨 후보를 추가로 확보할 수 있는지도 별도 확인한다. 이 표의 소켓 추정 좌표는 주 위치·각도 평가와 학습용 정답에서 제외한다.